# TP2 — Exercise 4

Run the cells from top to bottom. This notebook includes its prerequisite classes.

### Exercise 4: Transformation factory
Here, normalization means scaling numeric values to the range 0–1 (min-max scaling). Constant numeric columns become zero, and missing values remain missing. Text standardization removes surrounding spaces and uses lowercase. The factory chooses an object from a short string name.

In [1]:
import pandas as pd
from abc import ABC, abstractmethod
from functools import wraps
from time import perf_counter

#### Setup from earlier exercises

In [2]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None

    def read(self):
        self.data = pd.read_csv(self.file_path)
        return self.data

    def preview(self, n=5):
        if self.data is None:
            self.read()
        print(self.data.head(n))

In [3]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass


class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna().copy()


class FillMean(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.select_dtypes(include="number").columns:
            cleaned[column] = cleaned[column].fillna(cleaned[column].mean())
        return cleaned


class FillMode(MissingValueStrategy):
    def handle(self, df):
        cleaned = df.copy()
        for column in cleaned.columns:
            modes = cleaned[column].mode()
            if not modes.empty:
                cleaned[column] = cleaned[column].fillna(modes.iloc[0])
        return cleaned


class DataCleaner:
    def __init__(self, strategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

In [4]:
reader = CSVReader("Data/sample_data (2).csv")
raw_data = reader.read()

In [5]:
clean_data = DataCleaner(FillMean()).clean(raw_data)

#### Exercise 4 implementation

In [6]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass


class NormalizeColumns(DataTransform):
    def apply(self, df):
        transformed = df.copy()
        for column in transformed.select_dtypes(include="number").columns:
            values = transformed[column]
            minimum = values.min()
            spread = values.max() - minimum
            if spread == 0:
                transformed[column] = values.where(values.isna(), 0.0)
            else:
                transformed[column] = (values - minimum) / spread
        return transformed


class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates().copy()


class StandardizeText(DataTransform):
    def apply(self, df):
        transformed = df.copy()
        for column in transformed.select_dtypes(include=["object", "string"]).columns:
            transformed[column] = transformed[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return transformed


class TransformFactory:
    @staticmethod
    def create(name):
        transformations = {
            "normalize": NormalizeColumns,
            "remove_duplicates": RemoveDuplicates,
            "standardize_text": StandardizeText,
        }
        key = name.strip().lower()
        if key not in transformations:
            raise ValueError(f"Unknown transformation: {name}")
        return transformations[key]()

In [7]:
text_transform = TransformFactory.create("standardize_text")
print(text_transform.apply(clean_data).head())

   id     name    age  height_cm  weight_kg         city  score
0   1    alice  29.00      165.0       68.0     new york   85.0
1   2      bob  30.25      172.0       70.0  los angeles   90.0
2   3  charlie  35.00      168.0       72.0      chicago   86.0
3   4    david  30.25      167.0       80.0      houston   75.0
4   5      eva  27.00      160.0       55.0     new york   88.0
